In [1]:
import os
print(os.listdir("/kaggle/input/competitions"))

['global-wheat-detection']


In [2]:
import os
import ast
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image

import albumentations as A
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAME = "wheat"
CLASS_ID = 0

# ----- Data paths -----
# Adjust DATA_DIR to wherever the competition data was downloaded/extracted.
# Expected structure:
#   DATA_DIR/train.csv
#   DATA_DIR/train/*.jpg
#   DATA_DIR/test/*.jpg
DATA_DIR = Path("/kaggle/input/competitions/global-wheat-detection")          # e.g. Path("/kaggle/input/global-wheat-detection")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMG_DIR = DATA_DIR / "train"
TEST_IMG_DIR = DATA_DIR / "test"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
YOLO_DIR = OUTPUT_DIR / "yolo_dataset"

assert TRAIN_CSV.exists(), f"train.csv not found at {TRAIN_CSV.resolve()} - update DATA_DIR"
print("Using data directory:", DATA_DIR.resolve())

Using data directory: /kaggle/input/competitions/global-wheat-detection


In [3]:
all_train_images = sorted(p.stem for p in TRAIN_IMG_DIR.glob("*.jpg"))

def parse_bbox(bbox_str):
    """Parse the string-encoded bbox '[xmin, ymin, w, h]' into floats."""
    return ast.literal_eval(bbox_str)

df_raw = pd.read_csv(TRAIN_CSV)
df = df_raw.copy()
bbox_arr = np.array(df["bbox"].apply(parse_bbox).tolist())
df["x_min"] = bbox_arr[:, 0]
df["y_min"] = bbox_arr[:, 1]
df["box_width"] = bbox_arr[:, 2]
df["box_height"] = bbox_arr[:, 3]

images_with_boxes = set(df["image_id"].unique())
images_without_boxes = sorted(set(all_train_images) - images_with_boxes)

In [4]:
SMALL_AREA_RATIO_THRESH = 0.0005   # box covers < 0.05% of its image
LARGE_AREA_RATIO_THRESH = 0.15     # box covers > 15% of its image
df["area"] = df["box_width"] * df["box_height"]
df["image_area"] = df["width"] * df["height"]
df["area_ratio"] = df["area"] / df["image_area"]
NEG_DIM_MASK = (df["box_width"] <= 0) | (df["box_height"] <= 0)
SMALL_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] < SMALL_AREA_RATIO_THRESH)
LARGE_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] > LARGE_AREA_RATIO_THRESH)
NORMAL_MASK = ~(NEG_DIM_MASK | SMALL_MASK | LARGE_MASK)

In [5]:
clean_df = df.copy()
clean_df["x_max"] = clean_df["x_min"] + clean_df["box_width"]
clean_df["y_max"] = clean_df["y_min"] + clean_df["box_height"]

clean_df["is_negative_dim"] = NEG_DIM_MASK
clean_df["is_small_outlier"] = SMALL_MASK
clean_df["is_large_outlier"] = LARGE_MASK
clean_df["is_outlier"] = NEG_DIM_MASK | SMALL_MASK | LARGE_MASK
clean_df["use_for_training"] = ~(clean_df["is_outlier"])

attribute_cols = [
    "image_id", "width", "height", "source",
    "x_min", "y_min", "box_width", "box_height", "x_max", "y_max", "is_outlier",
    "use_for_training"
]
clean_df = clean_df[attribute_cols]

In [6]:
from sklearn.model_selection import train_test_split

# --- Build an 80/10/10 split, stratified by source ---
image_source_df = clean_df[["image_id", "source"]].drop_duplicates()

train_ids, temp_ids = train_test_split(
    image_source_df["image_id"], test_size=0.20,
    stratify=image_source_df["source"], random_state=RANDOM_SEED,
)
temp_source = image_source_df.set_index("image_id").loc[temp_ids, "source"]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_source, random_state=RANDOM_SEED,
)  # 0.5 of the 20% held out -> 10% val, 10% test

# images with no boxes have no known source - split them the same way, unstratified
no_box_ids = sorted(set(all_train_images) - set(image_source_df["image_id"]))
nb_train, nb_temp = train_test_split(no_box_ids, test_size=0.20, random_state=RANDOM_SEED)
nb_val, nb_test = train_test_split(nb_temp, test_size=0.50, random_state=RANDOM_SEED)

split_lookup = {}
for img_id in list(train_ids) + nb_train:
    split_lookup[img_id] = "train"
for img_id in list(val_ids) + nb_val:
    split_lookup[img_id] = "val"
for img_id in list(test_ids) + nb_test:
    split_lookup[img_id] = "test"

print(pd.Series(split_lookup).value_counts())

train    2737
test      343
val       342
Name: count, dtype: int64


In [7]:
YOLO_IMG_DIR = {s: YOLO_DIR / "images" / s for s in ["train", "val", "test"]}
YOLO_LBL_DIR = {s: YOLO_DIR / "labels" / s for s in ["train", "val", "test"]}
for d in list(YOLO_IMG_DIR.values()) + list(YOLO_LBL_DIR.values()):
    d.mkdir(parents=True, exist_ok=True)

def to_yolo_line(row, img_w, img_h):
    x_center = (row.x_min + row.box_width / 2) / img_w
    y_center = (row.y_min + row.box_height / 2) / img_h
    w = row.box_width / img_w
    h = row.box_height / img_h
    x_center, y_center, w, h = (float(np.clip(v, 0, 1)) for v in (x_center, y_center, w, h))
    return f"{CLASS_ID} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

boxes_for_yolo = clean_df[clean_df["use_for_training"]]

for img_id in all_train_images:
    split = split_lookup.get(img_id, "train")  # safe now: "train"/"val"/"test" all exist as keys
    rows = boxes_for_yolo[boxes_for_yolo["image_id"] == img_id]

    img_w, img_h = 1024, 1024
    if len(rows) > 0:
        img_w = int(rows.iloc[0]["width"])
        img_h = int(rows.iloc[0]["height"])

    lines = [to_yolo_line(r, img_w, img_h) for r in rows.itertuples()]
    (YOLO_LBL_DIR[split] / f"{img_id}.txt").write_text("\n".join(lines))

    src_img = TRAIN_IMG_DIR / f"{img_id}.jpg"
    dst_img = YOLO_IMG_DIR[split] / f"{img_id}.jpg"
    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except OSError:
            shutil.copy(src_img, dst_img)

data_yaml = f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: {CLASS_NAME}
"""
(YOLO_DIR / "data.yaml").write_text(data_yaml)

for split in ["train", "val", "test"]:
    n_img = len(list(YOLO_IMG_DIR[split].glob("*.jpg")))
    n_lbl = len(list(YOLO_LBL_DIR[split].glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")

train: 2737 images, 2737 labels
val  : 342 images, 342 labels
test : 343 images, 343 labels


In [8]:
# !pip install ultralytics -q
# from ultralytics import YOLO

# # Load the YOLO26s model
# model = YOLO("yolo26s.pt")

# # Train the model using the best hyperparameters from your tuning run
# results = model.train(
#     data="/kaggle/working/outputs/yolo_dataset/data.yaml",
#     epochs=50,         # Adjust epochs as needed for final training
#     imgsz=1024,         # Standard image size
#     batch=16,          # Adjust based on your GPU memory
#     project="global_wheat",
#     name="train_final",
#     exist_ok=True,
#     plots=True,        # Generates and saves training curves/plots
# )

In [11]:
!pip install ultralytics -q
from ultralytics import YOLO

# 1. UPGRADE TO LARGE MODEL (Maximum possible accuracy)
model = YOLO("yolo26l.pt") 

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,          
    imgsz=1024,          
    batch=8,             
    project="global_wheat",
    name="train_final_large", 
    exist_ok=True,
    plots=True,
    patience=15,         # Automatically stops if mAP@50-95 doesn't improve for 15 epochs
    
    # --- TWEAKS TO FOCUS ON mAP@50-95 ---
    box=8.5,             # INCREASED (Default is 7.5). Forces the model to care more about drawing tight, accurate boxes. 
                         # This directly improves mAP@75 and mAP@95.
    dfl=2.0,             # INCREASED (Default is 1.5). Distribution Focal Loss. Helps the model predict precise box boundaries.
    copy_paste=0.3,      # ADDED. This augmentation copies wheat heads and pastes them into new areas of the image. 
                         # It is a proven trick to significantly boost mAP for dense, small objects.
)

Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=8.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/outputs/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=2.0, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_final_large, nbs=64, nms=

In [12]:
from ultralytics import YOLO

# Load the best model weights from the MEDIUM model training run
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_large/weights/best.pt"
model = YOLO(model_path)

# Run validation specifically on the 'val' split
metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", split="val")

# Extract the core metrics from the Ultralytics results object
precision = metrics.box.mp      # Mean Precision
recall = metrics.box.mr         # Mean Recall
mAP_50 = metrics.box.map50      # mAP @ 0.50 IoU
mAP_75 = metrics.box.map75      # mAP @ 0.75 IoU
mAP_50_95 = metrics.box.map     # mAP @ 0.50:0.95 IoU

# Calculate F1 Score (Harmonic mean of Precision and Recall)
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Display the results cleanly
print("-" * 30)
print("   Validation Set Metrics   ")
print("-" * 30)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 Score:   {f1_score:.4f}")
print(f"mAP@50:     {mAP_50:.4f}")
print(f"mAP@75:     {mAP_75:.4f}")
print(f"mAP@50-95:  {mAP_50_95:.4f}")
print("-" * 30)

Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26l summary (fused): 188 layers, 24,746,511 parameters, 0 gradients, 86.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2864.0±337.8 MB/s, size: 191.9 KB)
val: Scanning /kaggle/working/outputs/yolo_dataset/labels/val.cache... 342 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 342/342 110.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.8s/it 40.6s1.9ss
                   all        342      14917      0.927      0.907      0.953      0.577
Speed: 4.2ms preprocess, 106.1ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
------------------------------
   Validation Set Metrics   
------------------------------
Precision:  0.9271
Recall:     0.9068
F1 Score:   0.9168
mAP@50:     0.9533
mAP@75:     0.6119
mAP@50-95:  0.5773
------------------------------


In [13]:
from ultralytics import YOLO

# Load your best model (YOLO26m performed better)
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_large/weights/best.pt"
model = YOLO(model_path)

# Run validation on the TEST split
print("=" * 50)
print("📊 TEST SET METRICS")
print("=" * 50)

metrics = model.val(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
    split="test",  # This validates on the test split
    imgsz=1024,
    batch=16,
    plots=True
)

# Extract and display metrics
precision = metrics.box.mp
recall = metrics.box.mr
mAP_50 = metrics.box.map50
mAP_75 = metrics.box.map75
mAP_50_95 = metrics.box.map
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "-" * 40)
print("   Test Set Performance   ")
print("-" * 40)
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1 Score:    {f1_score:.4f}")
print(f"mAP@50:      {mAP_50:.4f}")
print(f"mAP@75:      {mAP_75:.4f}")
print(f"mAP@50-95:   {mAP_50_95:.4f}  ← PRIMARY METRIC")
print("-" * 40)

📊 TEST SET METRICS
Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26l summary (fused): 188 layers, 24,746,511 parameters, 0 gradients, 86.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2775.5±903.1 MB/s, size: 175.0 KB)
val: Scanning /kaggle/working/outputs/yolo_dataset/labels/test... 343 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 343/343 1.1Kit/s 0.3s<0.0s
val: New cache created: /kaggle/working/outputs/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.9s/it 41.7s1.9ss
                   all        343      14498      0.926       0.91      0.955      0.583
Speed: 4.3ms preprocess, 108.6ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-2

----------------------------------------
   Test Set Performance   
----------------------------------------
Precision:   0.9261
Recall:     

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# from PIL import Image
# from pathlib import Path
# import random
# from ultralytics import YOLO

# # 1. Setup paths to your internal test set (which has labels)
# TEST_IMG_DIR = Path("/kaggle/working/outputs/yolo_dataset/images/test")
# TEST_LBL_DIR = Path("/kaggle/working/outputs/yolo_dataset/labels/test")

# # 2. Load your best trained model
# model_path = "/kaggle/working/runs/detect/global_wheat/train_final_medium/weights/best.pt"
# model = YOLO(model_path)

# # 3. Pick 4 random images from the test set
# all_test_imgs = list(TEST_IMG_DIR.glob("*.jpg"))
# selected_imgs = random.sample(all_test_imgs, 4)

# # Helper function to read YOLO format labels and convert to bounding boxes
# def get_yolo_boxes(lbl_path, img_w, img_h):
#     boxes = []
#     if lbl_path.exists():
#         with open(lbl_path, 'r') as f:
#             for line in f.readlines():
#                 parts = line.strip().split()
#                 if len(parts) == 5:
#                     cls, xc, yc, w, h = map(float, parts)
#                     # Convert YOLO (center_x, center_y, w, h) to (x_min, y_min, x_max, y_max)
#                     x1 = (xc - w/2) * img_w
#                     y1 = (yc - h/2) * img_h
#                     x2 = (xc + w/2) * img_w
#                     y2 = (yc + h/2) * img_h
#                     boxes.append([x1, y1, x2, y2])
#     return boxes

# # 4. Create the visualization grid (4 rows, 2 columns)
# fig, axes = plt.subplots(4, 2, figsize=(16, 24))
# fig.suptitle("Ground Truth vs Model Predictions (Test Set)", fontsize=20, fontweight='bold')

# for i, img_path in enumerate(selected_imgs):
#     img = Image.open(img_path)
#     img_w, img_h = img.size
#     img_id = img_path.stem
    
#     # Get Ground Truth boxes
#     lbl_path = TEST_LBL_DIR / f"{img_id}.txt"
#     gt_boxes = get_yolo_boxes(lbl_path, img_w, img_h)
    
#     # Get Model Predictions (using TTA for best results)
#     # In the prediction loop, change:
#     results = model.predict(
#         source=str(img_path),
#         imgsz=1024,
#         augment=True,
#         conf=0.25,      # Higher threshold
#         iou=0.45,       # Remove overlapping boxes
#         max_det=100,    # Limit max detections per image
#         verbose=False
#     )
#     pred_boxes = results[0].boxes.xyxy.cpu().numpy()
#     pred_confs = results[0].boxes.conf.cpu().numpy()
    
#     # --- Plot Ground Truth (Left Column) ---
#     ax_gt = axes[i, 0]
#     ax_gt.imshow(img)
#     ax_gt.set_title(f"Image {i+1}: Ground Truth ({len(gt_boxes)} boxes)", fontsize=14, color='green')
#     for box in gt_boxes:
#         x1, y1, x2, y2 = box
#         rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
#         ax_gt.add_patch(rect)
#     ax_gt.axis('off')
    
#     # --- Plot Predictions (Right Column) ---
#     ax_pred = axes[i, 1]
#     ax_pred.imshow(img)
#     ax_pred.set_title(f"Image {i+1}: Model Prediction ({len(pred_boxes)} boxes)", fontsize=14, color='red')
#     for j, box in enumerate(pred_boxes):
#         x1, y1, x2, y2 = box
#         conf = pred_confs[j]
#         rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
#         ax_pred.add_patch(rect)
#         # Add confidence score text
#         ax_pred.text(x1, y1-5, f"{conf:.2f}", color='red', fontsize=10, weight='bold', 
#                      bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
#     ax_pred.axis('off')

# plt.tight_layout()
# plt.show()